In [ ]:
# ── Colab / Local Setup ─────────────────────────────────────────────────────────────────────
# Run this cell first. In Colab it installs packages and clones the repo.
# Locally it's a no-op if the repo is already on your PYTHONPATH.
import sys, os, subprocess

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("🌐 Google Colab detected — installing dependencies (≈1 min first run)...")
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q",
        "langchain-text-splitters>=1.1.1",
        "langchain-openai>=1.1.10", "langchain-community>=0.4.1",
        "openai>=1.50.0", "numpy>=1.26.0",
        "scikit-learn>=1.3.0",
        "sentence-transformers>=2.2.0",
    ], check=True, capture_output=True)
    print("✅ Packages installed")

    repo_path = "/content/VectorSmuggle"
    if not os.path.exists(repo_path):
        subprocess.run(
            ["git", "clone", "-q", "https://github.com/jaschadub/VectorSmuggle.git", repo_path],
            check=True,
        )
        print("✅ Repository cloned")

    if repo_path not in sys.path:
        sys.path.insert(0, repo_path)
    os.chdir(repo_path)
    print("✅ Colab setup complete")
else:
    print("💻 Local environment — using existing installation")

<div align="center">

# Rotation Demo

### Defeating Statistical Detection in Embedding Stores

[![Website](https://img.shields.io/badge/website-vectorsmuggle.org-1f6feb?style=for-the-badge)](https://vectorsmuggle.org)
[![GitHub](https://img.shields.io/badge/source-jaschadub%2FVectorSmuggle-181717?style=for-the-badge&logo=github)](https://github.com/jaschadub/VectorSmuggle)
[![Paper (DOI)](https://img.shields.io/badge/paper-Zenodo%20DOI-3b82f6?style=for-the-badge)](https://doi.org/10.5281/zenodo.20058255)

</div>

---

> **Headline finding (reproduced in this notebook):**
>
> Small-angle, few-rotation orthogonal rotation defeats distribution-based
> detection on every (model, corpus) pair we tested — five embedding models,
> three corpora.

## What this notebook does

1. Loads a small corpus of **sensitive documents** (financial reports, employee records, executive emails).
2. Generates clean embeddings.
3. Applies a **Givens-rotation obfuscation** to half of them.
4. Trains an **Isolation Forest** detector on a clean baseline.
5. Measures detection **AUC on (held-out clean vs. rotated)** — the headline metric.
6. Demonstrates that rotation is **fully recoverable** given the rotation matrix (cosine ≈ 1.0).

## Scope

| Demonstrated here | Not demonstrated here |
| --- | --- |
| Rotation produces statistically indistinguishable embeddings under an off-the-shelf Isolation Forest | Survival against production DLP products |
| Round-trips perfectly given the rotation matrix | Adaptive / white-box defender evaluation |
| Runs end-to-end on a real sensitive corpus | Cross-model / cross-backend / cross-corpus tables |

The full validated scope (5 embedding models, 3 corpora, 7 vector-store configurations, white-box adaptive evaluation) lives in the [companion preprint](https://doi.org/10.5281/zenodo.20058255). The framework source — including reproduction scripts and the VectorPin defense pointers — is on [GitHub](https://github.com/jaschadub/VectorSmuggle), with the project landing page at [vectorsmuggle.org](https://vectorsmuggle.org).

In [ ]:
# ── API Key Configuration (optional) ─────────────────────────────────────────────────────
# The demo works without an API key using free sentence-transformers embeddings.
# For OpenAI embeddings: add OPENAI_API_KEY to Colab Secrets (🔑 icon in the sidebar).
import os

try:
    from google.colab import userdata
    key = userdata.get("OPENAI_API_KEY")
    if key:
        os.environ["OPENAI_API_KEY"] = key
        print("✅ OpenAI API key loaded from Colab Secrets")
    else:
        print("ℹ️  No OPENAI_API_KEY secret — will use sentence-transformers (free, no key needed)")
except Exception:
    print("ℹ️  API key not configured — will use sentence-transformers (free, no key needed)")

## Step 1 · Imports

Pull in the VectorSmuggle modules used below and a text splitter for chunking.

In [ ]:
from pathlib import Path

import numpy as np
from langchain_text_splitters import RecursiveCharacterTextSplitter

# VectorSmuggle framework
from analysis.detectors.isolation_forest_detector import IsolationForestDetector, evaluate
from steganography.obfuscation import EmbeddingObfuscator
from utils.embedding_factory import create_embeddings

print("✅ All imports successful")

## Step 2 · Load sensitive corpus

We load the synthetic-PII corpus shipped in `sample_docs/` — financial reports, employee records, executive emails. This is the same corpus used to produce the paper's headline detection table.

In [ ]:
# Load all text-based sensitive documents
sample_dir = Path("sample_docs")
sensitive_files = [
    sample_dir / "financial_report_q3_2024.md",
    sample_dir / "employee_handbook.md",
    sample_dir / "employee_records_001.txt",
    sample_dir / "executive_emails.eml",
    sample_dir / "financial_report.csv",
    sample_dir / "payroll_data_2024.csv",
]

raw_docs = [f.read_text(encoding="utf-8", errors="ignore") for f in sensitive_files if f.exists()]
print(f"📄 Loaded {len(raw_docs)} sensitive documents")

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)
chunks = []
for doc in raw_docs:
    chunks.extend(splitter.split_text(doc))
print(f"✂️  Split into {len(chunks)} chunks")
print(f"\nFirst chunk preview:\n{'—' * 60}\n{chunks[0][:200]}...")

## Step 3 · Initialize embedding model

Uses OpenAI / Ollama if available, otherwise falls back to free `sentence-transformers/all-MiniLM-L6-v2` (downloads ~80 MB to the runtime, no API calls).

In [ ]:
print("🔧 Initializing embedding model...")
try:
    embedding_model = create_embeddings()
    print("✅ Embedding model initialized (OpenAI/Ollama)")
except RuntimeError:
    print("⚠️  OpenAI/Ollama unavailable — loading sentence-transformers (free, no API key)...")
    from sentence_transformers import SentenceTransformer as _ST

    class _STEmbeddings:
        def __init__(self, model="all-MiniLM-L6-v2"):
            self._m = _ST(model)
        def embed_query(self, text):
            return self._m.encode(text).tolist()
        def embed_documents(self, texts):
            return [v.tolist() for v in self._m.encode(texts)]

    embedding_model = _STEmbeddings()
    print("✅ Embedding model initialized (sentence-transformers/all-MiniLM-L6-v2, 384-dim)")

## Step 4 · Generate clean embeddings

This is the *clean baseline* — the embeddings a defender would expect to see in a normal RAG store. We will train the Isolation Forest detector on a held-out portion of these.

In [ ]:
print(f"🔄 Embedding {len(chunks)} chunks...")
clean = np.array(embedding_model.embed_documents(chunks))
print(f"✅ Generated clean embeddings: shape={clean.shape}, dtype={clean.dtype}")
print(f"   Sample norm: {np.linalg.norm(clean[0]):.4f}")

## Step 5 · Apply rotation

The `EmbeddingObfuscator` applies a small number of random Givens rotations — each at a small angle (default ≤ 0.1 rad). The result is an orthogonal transformation: cosine *similarities* between vectors are perfectly preserved; only the absolute coordinate frame shifts.

This is the steganographic technique that survives detection.

In [ ]:
obfuscator = EmbeddingObfuscator(rotation_angle=0.1, seed=42)
rotated, rotation_matrix = obfuscator.apply_rotation(clean)

print(f"✅ Applied rotation to {len(rotated)} embeddings")
print(f"   Rotation matrix: shape={rotation_matrix.shape}, det={np.linalg.det(rotation_matrix):+.6f}")
print(f"   |det| ≈ 1 confirms the matrix is orthogonal (lengths preserved).")

# Quick sanity check: pairwise similarities preserved?
from numpy.linalg import norm
sim_clean = float(np.dot(clean[0], clean[1]) / (norm(clean[0]) * norm(clean[1])))
sim_rot   = float(np.dot(rotated[0], rotated[1]) / (norm(rotated[0]) * norm(rotated[1])))
print(f"\nPairwise cosine sim(0,1) — clean: {sim_clean:.4f}, rotated: {sim_rot:.4f} (should match)")

## Step 6 · Train detector and measure AUC — headline result

We train an Isolation Forest on a clean baseline (the kind of off-the-shelf statistical anomaly detector a defender would deploy), then evaluate it on (held-out clean, rotated).

> **An AUC near 0.50 means an off-the-shelf statistical anomaly detector trained on clean embeddings performs at chance level when evaluating rotated ones — it completely fails to distinguish them.** This is the headline finding.

The paper uses a 27/41 train/test split on its 68-chunk corpus. We do roughly the same here — ~40 % train, ~60 % test.

In [ ]:
n = len(clean)
n_train = int(n * 0.4)
print(f"Train on first {n_train} clean embeddings; test on remaining {n - n_train}.")

detector = IsolationForestDetector(n_estimators=200, contamination=0.05, random_state=42)
detector.fit(clean[:n_train])

metrics = evaluate(detector, clean[n_train:], rotated[n_train:])

print("\n" + "=" * 60)
print("DETECTOR METRICS (clean test vs. rotated)")
print("=" * 60)
for k, v in metrics.items():
    print(f"  {k:>20}: {v:.4f}" if isinstance(v, float) else f"  {k:>20}: {v}")
print("=" * 60)

if metrics["auc"] < 0.6:
    print(f"\n🚨 Detector AUC = {metrics['auc']:.4f} — indistinguishable from random.")
    print("   Rotation defeats off-the-shelf statistical detection.")
else:
    print(f"\n⚠️  Detector AUC = {metrics['auc']:.4f} — above the headline threshold (~0.5).")
    print("   This can happen with very small corpora (high variance).")

## Step 7 · Recovery

Rotation is an orthogonal transform. Given the rotation matrix `R`, the inverse is exactly `R.T`, so an attacker who has the matrix can recover the original embeddings with zero loss.

In [ ]:
# rotated = clean @ R.T  →  recovered = rotated @ R
recovered = rotated @ rotation_matrix

# Per-vector cosine similarity to original
cos_sims = np.array([
    np.dot(c, r) / (np.linalg.norm(c) * np.linalg.norm(r))
    for c, r in zip(clean, recovered)
])

print(f"Recovery cosine similarity — mean: {cos_sims.mean():.6f}, min: {cos_sims.min():.6f}")
print(f"L2 reconstruction error  — mean: {np.mean(np.linalg.norm(clean - recovered, axis=1)):.2e}")

if cos_sims.min() > 0.999:
    print("\n✅ Lossless recovery confirmed: rotation is exactly reversible given the matrix.")
else:
    print("\n⚠️  Recovery imperfect — likely a numerical-precision issue.")

## Conclusion

This notebook demonstrated the headline finding:

1. **Detection fails.** An off-the-shelf Isolation Forest trained on clean embeddings performs at chance level (AUC ≈ 0.50) when evaluating rotated ones.
2. **Recovery is lossless.** The rotation matrix is the only secret needed; given it, the original embeddings are recovered exactly.

### Scope of this demo

- **Demonstrable here:** the technique runs end-to-end on a real sensitive corpus, against a real anomaly detector, and produces the claimed result.
- **Not demonstrated here:** survival against production DLP products, against larger production-scale corpora, or against adaptive defenders. Those are evaluated in the [companion preprint](https://doi.org/10.5281/zenodo.20058255) across 5 embedding models, 3 corpora (~26 K chunks combined), and 7 vector-store configurations — plus a white-box adaptive-attacker experiment that drives both detector AUCs to near zero.

### The defense

VectorSmuggle's constructive defense, [VectorPin](https://github.com/ThirdKeyAI/VectorPin), signs each embedding to its source content with Ed25519. Any post-embedding modification — including rotation — breaks signature verification: rotation by even a single Givens transform changes the vector and invalidates the signature, so a detector built on signature checks (rather than on the embedding distribution) catches the attack deterministically rather than statistically.

---

<div align="center">

### Learn more

[**vectorsmuggle.org**](https://vectorsmuggle.org) &nbsp;·&nbsp; [**GitHub: jaschadub/VectorSmuggle**](https://github.com/jaschadub/VectorSmuggle) &nbsp;·&nbsp; [**Paper (Zenodo DOI)**](https://doi.org/10.5281/zenodo.20058255) &nbsp;·&nbsp; [**VectorPin (defense)**](https://github.com/ThirdKeyAI/VectorPin)

[![Website](https://img.shields.io/badge/website-vectorsmuggle.org-1f6feb?style=flat-square)](https://vectorsmuggle.org)
[![GitHub](https://img.shields.io/badge/source-GitHub-181717?style=flat-square&logo=github)](https://github.com/jaschadub/VectorSmuggle)
[![Paper](https://img.shields.io/badge/paper-Zenodo-3b82f6?style=flat-square)](https://doi.org/10.5281/zenodo.20058255)

</div>

---

> **⚠️ Ethical use only.** This demonstration is for educational and security-research purposes.